<a href="https://colab.research.google.com/github/andreaprifti19/SINDy-Teaching-Usecases/blob/main/notebooks/02_lorenz_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Use Case 2: Discovering the Equations of the Lorenz System with SINDy

In Use Case 1 we let SINDy find the equations of a damped oscillator. That system was simple: two variables, only linear terms, and a smooth spiral towards rest.

Now we take the next step. The **Lorenz system** has three variables, **nonlinear** terms, and it is **chaotic**: its motion never repeats and never settles down. It is the classic test system of the original SINDy paper (Brunton et al., 2016), and it shows clearly where equation discovery becomes hard.

We use exactly the same workflow as in Use Case 1, so you can compare the two notebooks step by step.

**What is new compared to Use Case 1**

- Three variables and nonlinear terms ($xz$, $xy$), so the candidate library must contain products of variables.
- Chaos: a correct model still drifts away from the data after some time. We need new ways to check if a model is correct.
- A new tool, **Ensemble-SINDy**, which tells us how much we can trust each discovered term.

**What you will learn**

1. Why the Lorenz system is a harder test for SINDy than the oscillator.
2. How to run SINDy on a nonlinear system with three variables.
3. How to validate a model of a chaotic system, when comparing curves point by point no longer works.
4. When SINDy fails: noise, too much smoothing, too little data, or an unsuitable library.

**How to use this notebook**

- Run the cells **from top to bottom**, in order. Later cells use variables from earlier cells.
- If you change something, use *Runtime → Restart and run all* so that every result comes from the same run.
- All randomness uses a fixed seed, so you should see exactly the same numbers as written in the text.

**Contents**

1. Setup
2. The physical system
3. Generating the data
4. Discovering the equations with SINDy
5. Validation: is the discovered model correct?
6. Robustness experiments: noise, smoothing, Ensemble-SINDy, data quantity, candidate library
7. Summary
8. Exercises

## 1. Setup

We install the same PySINDy version as in Use Case 1 (`2.1.0`), so both notebooks behave the same.

In [1]:
!pip install pysindy==2.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.4/123.4 kB 3.6 MB/s eta 0:00:00


The imports and the reproducibility settings are the same as in Use Case 1, with one addition:

- **Warnings.** We hide the same two numerical warnings as before. In addition, with very noisy data STLSQ sometimes removes *every* term and then prints a warning. In our tables such a fit simply counts as a failure, so we hide this one message too. All other warnings are still shown.
- **Random seed.** `SEED = 0` makes the random noise identical on every run and every computer. (Section 6 explains how every noisy dataset is built from this seed.)

In [2]:
import warnings

import numpy as np
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.linalg import LinAlgWarning
from sklearn.metrics import mean_absolute_error
from sklearn.exceptions import ConvergenceWarning
import pysindy as ps

# Same two warnings as in Use Case 1 (they appear often in the robustness sweeps).
warnings.filterwarnings("ignore", category=LinAlgWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
# Very noisy fits sometimes lose every term. We count those as failures,
# so the warning itself is not needed.
warnings.filterwarnings("ignore", message="Sparsity parameter is too big")

# Fixed seed -> identical noise, identical numbers, on every run and every machine.
SEED = 0

print("pysindy      ", ps.__version__)
print("numpy        ", np.__version__)
print("scipy        ", scipy.__version__)
print("scikit-learn ", sklearn.__version__)
print("matplotlib   ", matplotlib.__version__)

pysindy       2.1.0
numpy         2.1.3
scipy         1.16.3
scikit-learn  1.6.1
matplotlib    3.10.0


All figures are saved into the same `figures/` folder as in Use Case 1. The file names start with `lorenz_`, so they do not overwrite the figures of the first notebook.

In [3]:
import os
# In Colab: saves to "figures/". Run locally from the repo's notebooks/ folder: saves to the repo's figures/.
FIG_DIR = "../figures" if os.path.isdir("../figures") else "figures"
os.makedirs(FIG_DIR, exist_ok=True)